# 🦾 Training Toolkit: Fine-tune Llava-NeXT-Video for JSON output

- **Input**: video, text prompt saying "extract JSON"
- **Output**: JSON object with information extracted from the video

In [ ]:
from dotenv import load_dotenv  
from pathlib import Path
import sys

sys.path.append(Path("..").resolve().as_posix())  # add the root of the project to the path to enable imports
_ = load_dotenv()

## Easy mode 🙈

**To get started**, convert your dataset to HF 🤗 Datasets format and save it locally.

It needs to have two columns:

- `video_path` column that contains a relative path to video relative.
- `json` column that contains a **string** representation of your target JSON.


In [ ]:
DATASET_PATH = "path/to/dataset"

In [ ]:
from training_toolkit import build_trainer, llava_next_video_preset, video_json_preset

In [ ]:
trainer = build_trainer(
    **llava_next_video_preset.as_kwargs(),
    **video_json_preset.with_path(DATASET_PATH).as_kwargs(),
)

In [ ]:
trainer.train()

## Advanced mode 🙉

Preset parameters don't quite fit your case? Doesn't mean you need to build *everything* from scratch.

Learn how to combine Training Toolkit with custom tools to minimize time and energy required to train your adapter.

### 1. Tune preset parameters

Both `llava_next_video_preset` and `video_json_preset` are Pydantic dataclasses.

- Both return a dict when you call `as_kwargs()` on them.
- Combined, those dicts contain all parameters necessary to train an adapter.
- You can edit those parameters directly as dataclass attributes.

In [ ]:
from training_toolkit import build_trainer, llava_next_video_preset, video_json_preset

In [ ]:
# let's take a look at the arguments build_trainer expects

?build_trainer

In [ ]:
# we can find all these argumets in the presets

print(f"model preset: {llava_next_video_preset.as_kwargs().keys()}")
print(f"data preset args: {video_json_preset.with_path("path/to/dataset").as_kwargs().keys()}")

In [ ]:
# let's edit a couple hyperparameters 

llava_next_video_preset.hf_model_id="lmms-lab/LLaVA-NeXT-Video-7B-DPO"

llava_next_video_preset.training_args["per_device_train_batch_size"] = 24
llava_next_video_preset.training_args["per_device_eval_batch_size"] = 24
llava_next_video_preset.training_args["eval_strategy"] = "no"
llava_next_video_preset.training_args["num_train_epochs"] = 5

In [ ]:
# now as_kwargs will return updated arguments

llava_next_video_preset.as_kwargs()

### 2. Customize dataset format

Say your dataset doesn't match the strict criteria described above. Or you have your own idea of what exactly the inputs and the targets should look like.

In order to use your dataset with the rest of the toolkit, you need to write **your own data collator**. This is a utility that turns a list of individual samples from your dataset into a batch ready to go into the model. Those batches are complete with padded tokenized text, attention masks and preprocessed video.

**Note**: Video support in HF 🤗 frameworks is... Not thorough. By default, Training Toolkit loads and properly arranges videos as part of the collator logic.

When writing a custom collator we need to jump through the same set of hoops in order to get our videos loaded, preprocessed and properly padded.

The HF 🤗 Transformers processor (see HF docs) is going to do the rest. To summarize, we need to do the following:

1. Load videos from video paths and convert them to tensors
2. Prepare text prompt. It contains both the input part and the bits we want to teach the model to generate
4. Feed each individual sample into the processor
5. Pad and concatenate samples together to produce a single batch

In [ ]:
from training_toolkit import DataPreset
from training_toolkit.common.video_readers import get_video_reader
import torch
import os
import json


class SpecialVideoJSONCollator:
    def __init__(self, processor, num_frames=8, max_length=256):
        self.processor = processor

        self.num_frames = num_frames
        self.max_length = max_length

        self.num_proc = os.cpu_count()
        self.read_video_fn = get_video_reader()

    def __call__(self, examples):
        samples = []
        for example in examples:

            # 1. Load video
            video = torch.tensor(
                self.read_video_fn(
                    example["video"],
                    self.num_frames,
                )
            )

            # 2. Prepare the prompt
            conversation = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "extract JSON."},
                        {"type": "video"},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": json.dumps(example["json"])},
                    ],
                },
            ]

            prompt = self.processor.apply_chat_template(
                conversation, add_generation_prompt=False
            )

            # 3. Process the video and tokenize the prompt
            sample = self.processor(
                text=prompt,
                videos=video,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
            samples.append(sample)

        # 4. Pad the inputs
        padded_inputs = self.processor.tokenizer.pad(
            {
                "input_ids": [
                    sample["input_ids"][0] for sample in samples
                ],  # each element is one batch only so we slice [0]
                "attention_mask": [sample["attention_mask"][0] for sample in samples],
            },
            padding=True,
            return_tensors="pt",
        )

        labels = padded_inputs["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        padded_inputs["labels"] = labels

        # 5. Put the videos back in
        padded_inputs["pixel_values_videos"] = torch.cat(
            [sample["pixel_values_videos"] for sample in samples], dim=0
        )
        return padded_inputs


# let's wrap the new collator into a custom DataPreset
special_video_json_preset = DataPreset(
    train_test_split=0.1,
    collator_cls=SpecialVideoJSONCollator,
)

# finally, we can make sure everything loads correctly
special_video_json_preset.with_path("path/to/data").as_kwargs()